# Minimal working example using BERT (`distBERT` model)

In [1]:
#link with drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
items = pd.read_csv('/content/drive/My Drive/Data_Eng_300/items.csv')
events = pd.read_csv('/content/drive/My Drive/Data_Eng_300/events.csv')

In [5]:
events["ts"] = pd.to_datetime(events["ts"])

In [6]:
# items = pd.read_csv('items.csv')
# events = pd.read_csv('events.csv', parse_dates=['ts'])

In [7]:
items.head()

,item_id,title,description,category,brand,price,tags
0,i1,Noise Cancelling Headphones,Wireless noise-cancelling headphones with 30-h...,electronics,SoundPeak,199.99,"audio,wireless,travel,premium"
1,i2,Mechanical Keyboard,"Mechanical keyboard with RGB backlighting, hot...",electronics,KeyForge,129.00,"keyboard,gaming,productivity,desk"
2,i3,Running Shoes,Running shoes designed for long distance comfo...,sports,StrideLab,110.00,"running,fitness,outdoors,comfort"
3,i4,Vegetarian Cookbook,"Cookbook featuring quick vegetarian recipes, p...",books,HomeTable Press,24.99,"cooking,vegetarian,recipes,home"
4,i5,Fitness Smartwatch,"Smartwatch with heart rate monitoring, GPS, sl...",electronics,PulsePath,249.00,"wearables,fitness,gps,health"


In [8]:
events.head()

,user_id,item_id,ts,event_type,session_id,device,dwell_seconds,price_at_event,category_at_event
0,u1,i2,2026-01-01 07:47:00,click,s_u1_20260101_1,tablet,56,129.00,electronics
1,u1,i13,2026-01-04 08:01:00,view,s_u1_20260104_2,tablet,7,149.99,electronics
2,u1,i2,2026-01-07 14:28:00,click,s_u1_20260107_3,mobile,72,129.00,electronics
3,u1,i13,2026-01-07 19:06:00,view,s_u1_20260107_1,tablet,67,149.99,electronics
4,u1,i1,2026-01-10 08:42:00,add_to_cart,s_u1_20260110_1,desktop,46,199.99,electronics


In [2]:
!pip install pandas torch transformers faiss-cpu

## Setup

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss

## Pseudo data creation

In [9]:
# # ----------------------------
# # 1) Sample pseudo "data"
# # ----------------------------
# items = pd.DataFrame([
#     {"item_id":"i1","title":"Noise Cancelling Headphones","description":"Wireless noise-cancelling headphones with 30-hour battery life","category":"electronics"},
#     {"item_id":"i2","title":"Mechanical Keyboard","description":"Mechanical keyboard with RGB and hot-swappable switches","category":"electronics"},
#     {"item_id":"i3","title":"Running Shoes","description":"Running shoes designed for long distance comfort and stability","category":"sports"},
#     {"item_id":"i4","title":"Vegetarian Cookbook","description":"Cookbook featuring quick vegetarian recipes for busy weeknights","category":"books"},
#     {"item_id":"i5","title":"Fitness Smartwatch","description":"Smartwatch with heart rate monitoring, GPS, and sleep tracking","category":"electronics"},
# ])

# events = pd.DataFrame([
#     {"user_id":"u1","item_id":"i1","ts":"2026-01-10"},
#     {"user_id":"u1","item_id":"i5","ts":"2026-01-11"},
#     {"user_id":"u2","item_id":"i2","ts":"2026-01-12"},
#     {"user_id":"u3","item_id":"i3","ts":"2026-01-13"},
#     {"user_id":"u3","item_id":"i4","ts":"2026-01-14"},
# ])
# events["ts"] = pd.to_datetime(events["ts"])


## BERT encoder (What makes BERT work?)

In [10]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilbert-base-uncased"  # for illustration, 66M model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Evaluate the BERT encoder**

In [11]:
# ----------------------------
# 2) BERT encode helper
# ----------------------------

# turn off gradient calculation (no back propagation in using BERT)
@torch.no_grad()
def bert_embed(texts, max_len=128):
    batch = tokenizer(
        texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    out = encoder(**batch)
    cls = out.last_hidden_state[:, 0]          # first column [CLS]-like token for classification
    emb = F.normalize(cls, dim=-1)             # normalization
    return emb.cpu().numpy().astype("float32") # size (B, 768)


## Embedding items into vectors (for comparisons)

In [12]:
# ----------------------------
# 3) Offline job: item embeddings
# ----------------------------
items["text"] = (
    items["title"].fillna("") + ". " +
    items["description"].fillna("") + " " +
    items["category"].fillna("") + " " +
    items["brand"].fillna("") + " " +
    items["price"].fillna("").astype(str) + " " +
    items["tags"].fillna("")
).str.replace(r"\s+", " ", regex=True).str.strip()

item_vecs = bert_embed(items["text"].tolist())
item_id_list = items["item_id"].tolist()

# Build ANN index (inner product works with normalized vectors)
index = faiss.IndexFlatIP(item_vecs.shape[1])
index.add(item_vecs)


## What user-specific data are there?

In [13]:
# ----------------------------
# 4) Feature builder: user text from last N clicks
# ----------------------------
# def build_user_text(user_id, events, items, N=3):
#     hist = (
#         events[(events["user_id"] == user_id) & (events["event_type"] == "click")]
#         .sort_values("ts")
#         .tail(N)["item_id"]
#         .tolist()
#     )

#     if not hist:
#         return "no history", set()

#     item_text = items.set_index("item_id")["text"]
#     text = item_text.reindex(hist).dropna().tolist()
#     if not text:
#         return "no history", set(hist)

#     return " ".join(text), set(hist)

#user clicks same item multiple times, we only keep most recent click
def build_user_text(user_id, events, items, N=3):
    clicks = events[(events["user_id"] == user_id) & (events["event_type"] == "click")]
    if clicks.empty:
        return "no history", set()

    hist = clicks.sort_values("ts")["item_id"].tolist()

    #keep last N unique item_ids to preserve recency
    last_unique = []
    for iid in reversed(hist):
        if iid not in last_unique:
            last_unique.append(iid)
        if len(last_unique) == N:
            break
    last_unique = list(reversed(last_unique))

    item_text = items.set_index("item_id")["text"]
    text_parts = item_text.reindex(last_unique).dropna().tolist()

    return (" ".join(text_parts) if text_parts else "no history"), set(last_unique)


## How to recommend "similar" item with BERT?

In [14]:
# ----------------------------
# 5) "What to recommend" function
# ----------------------------
# def recommend(user_id, k=3):
#     user_text, seen = build_user_text(user_id, events, items, N=3)
#     u = bert_embed([user_text])  # (1, 768)
#     scores, idx = index.search(u, k + len(seen))  # keep track of what was seen by the user
#     recs = []
#     for j in idx[0]:
#         iid = item_id_list[j]
#         if iid not in seen:
#             recs.append(iid)
#         if len(recs) == k:
#             break
#     return recs

def recommend(user_id, k=3, N=3):
    user_text, seen = build_user_text(user_id, events, items, N=N)
    u = bert_embed([user_text])

    #few extra to account for filtering(seen/missing)
    k_search = min(len(item_id_list), k + len(seen) + 10)
    scores, idx = index.search(u, k_search)

    recs = []
    for j in idx[0]:
        if j < 0:   #when not enough neighbors
            continue
        iid = item_id_list[j]
        if iid in seen:
            continue
        recs.append(iid)
        if len(recs) == k:
            break

    return recs

In [15]:

for u in events["user_id"].unique()[:10]:
    print(u, "->", recommend(u, k=3))


u1 -> ['i3', 'i6', 'i5']
u10 -> ['i8', 'i9', 'i18']
u2 -> ['i18', 'i17', 'i13']
u3 -> ['i3', 'i18', 'i5']
u4 -> ['i4', 'i20', 'i8']
u5 -> ['i5', 'i13', 'i18']
u6 -> ['i18', 'i4', 'i16']
u7 -> ['i8', 'i20', 'i18']
u8 -> ['i18', 'i5', 'i17']
u9 -> ['i16', 'i20', 'i18']


In [ ]:
import nbformat
nb = nbformat.read("items_events_yegon.ipynb", as_version=nbformat.NO_CONVERT)
if "widgets" in nb.metadata and "state" not in nb.metadata.widgets:
    nb.metadata["widgets"] = {"state": {}}
nbformat.write(nb, "items_events_yegon.ipynb")

FileNotFoundError: [Errno 2] No such file or directory: 'your_notebook.ipynb'